## Constraints 


In [1]:
import numpy as np
from dataclasses import dataclass


@dataclass
class Obstacle:
    """
    Represents a spherical obstacle that can move.
    """
    center: np.ndarray  # [x, y, z] position
    velocity: np.ndarray  # [vx, vy, vz] velocity
    radius: float  # Radius of the obstacle

    def get_cbf_constraints(self, drone_pos, drone_vel, k0=5.0, k1=5.0):
        """
        Calculates the linear constraint on control input u:
        A * u <= b
        Derived from the High-Order CBF condition:
        h_ddot + k1 * h_dot + k0 * h >= 0
        """
        # Relative State
        # We care about the vector pointing from Obstacle -> Drone
        rel_pos = drone_pos - self.center
        rel_vel = drone_vel - self.velocity

        # Barrier Function h(x)
        # h = ||dp||^2 - R^2
        dist_sq = np.dot(rel_pos, rel_pos)
        h = dist_sq - self.radius ** 2

        # First Derivative h_dot
        # h_dot = 2 * dp^T * dv
        h_dot = 2 * np.dot(rel_pos, rel_vel)

        # Second Derivative h_ddot terms
        # h_ddot = 2*||dv||^2 + 2*dp^T * (u_drone - u_obs)
        # We assume obstacle acceleration (u_obs) is 0 for prediction.

        # The constraint is:
        # 2*dp^T * u_drone >= -2*||dv||^2 - k1*h_dot - k0*h

        # Convert to form: A * u <= b
        A = -2 * rel_pos
        b = 2 * np.dot(rel_vel, rel_vel) + k1 * h_dot + k0 * h
        return A, b


## Dynamics

In [2]:
import numpy as np
from dataclasses import dataclass


@dataclass
class State:
    """
    Represents the full 6D state of the drone.
    We separate position and velocity for clarity.
    """
    pos: np.ndarray  # [x, y, z] (Meters)
    vel: np.ndarray  # [vx, vy, vz] (Meters/second)

    @property
    def vector(self):
        """Returns the full 6x1 state vector used for matrix math."""
        return np.concatenate([self.pos, self.vel])


class DroneDynamics:
    def __init__(self, dt: float = 0.05):
        self.dt = dt

        # --- The Double Integrator Matrices ---
        # State Transition Matrix A (6x6)
        # p_next = p + v*dt
        # v_next = v
        self.A = np.eye(6)
        self.A[0:3, 3:6] = np.eye(3) * dt

        # Control Input Matrix B (6x3)
        # p_next += 0
        # v_next += u*dt
        self.B = np.zeros((6, 3))
        self.B[3:6, :] = np.eye(3) * dt

    def step(self, state: State, u: np.ndarray) -> State:
        """
        Advances the simulation by one time step dt.
        x_{k+1} = A * x_k + B * u_k
        """
        x_k = state.vector

        # Apply the discrete dynamics
        x_next = self.A @ x_k + self.B @ u

        # Return a new State object
        return State(pos=x_next[:3], vel=x_next[3:])



## Guidance

In [3]:
import numpy as np
import scipy.linalg


class NominalGuidance:
    """
    Generates a nominal control input u_nom to track a target.
    Uses LQR (derived from CLF theory) to find the optimal guidance.
    """

    def __init__(self):
        # Define the System (Double Integrator Error Dynamics)
        # e_dot = A*e + B*u
        A = np.zeros((6, 6))
        A[0:3, 3:6] = np.eye(3)

        B = np.zeros((6, 3))
        B[3:6, :] = np.eye(3)

        # Define Weights (Tuning Knobs)
        # Q: Penalty on position/velocity error
        # R: Penalty on control effort
        Q = np.eye(6) * 10.0
        R = np.eye(3) * 1.0

        # Solve ARE for P
        P = scipy.linalg.solve_continuous_are(A, B, Q, R)

        # Compute LQR Gain Matrix K = R^-1 * B^T * P
        # This gives the optimal control law: u = -K * error
        self.K = np.linalg.inv(R) @ B.T @ P

    def compute_u_nom(self, drone_pos, drone_vel, target_pos, target_vel):
        """
        Calculates u_nom based on current error state.
        """
        err_pos = drone_pos - target_pos
        err_vel = drone_vel - target_vel
        error_state = np.concatenate([err_pos, err_vel])

        # u = -K * e
        u_nom = -self.K @ error_state
        noise = np.random.normal(0, 0.01, size=3)
        return u_nom + noise

## Safety Filter

In [4]:
import numpy as np
import osqp
from scipy import sparse


class SafetyFilter:
    def __init__(self, u_max=10.0):
        self.u_max = u_max

    def filter(self, u_nom, drone_pos, drone_vel, obstacles):
        """
        Solves QP:
        we use the identity matrix for Hessian matrix
        minimize || u - u_nom ||^2
        subject to CBF Constraints (A_cbf * u <= b_cbf)
        """

        # --- SETUP QP ---
        # Minimize (1/2)u^T P u + q^T u
        # To match ||u - u_nom||^2, we expand to: u^T u - 2*u_nom^T u
        # So P (Hessian) is Identity, q (linear) is -u_nom

        P_qp = sparse.csc_matrix(np.eye(3))
        q_qp = -u_nom

        # --- OBSTACLE CONSTRAINTS ---
        A_cbf_list = []
        b_cbf_list = []

        for obs in obstacles:
            A_i, b_i = obs.get_cbf_constraints(drone_pos, drone_vel)
            A_cbf_list.append(A_i)
            b_cbf_list.append(b_i)

        # --- BUILD CONSTRAINTS ---
        if not A_cbf_list:
            # No obstacles
            return np.clip(u_nom, -self.u_max, self.u_max)

        A_cons = np.vstack(A_cbf_list)
        b_cons = np.hstack(b_cbf_list)

        # Add Actuator Box Constraints (-u_max <= u <= u_max)
        A_box = np.eye(3)
        l_box = np.full(3, -self.u_max)
        u_box = np.full(3, self.u_max)

        # Combine for OSQP (l <= Ax <= u)
        # CBF is one-sided (Ax <= b), so l = -inf
        l_cons = np.full_like(b_cons, -np.inf)

        A_final = sparse.csc_matrix(np.vstack([A_cons, A_box]))
        l_final = np.hstack([l_cons, l_box])
        u_final = np.hstack([b_cons, u_box])

        # --- SOLVE ---
        prob = osqp.OSQP()
        prob.setup(P_qp, q_qp, A_final, l_final, u_final, verbose=False, polish=False)
        res = prob.solve()

        # --- STATUS ---
        if res.info.status not in ['solved', 'solved inaccurate']:
            print("Safety Filter Infeasible! Braking.")
            norm_v = np.linalg.norm(drone_vel)
            if norm_v > 0.01:
                # Direction opposing velocity
                u_brake = -drone_vel / norm_v * self.u_max
                return u_brake
            else:
                return np.zeros(3)

        return res.x


## Scenario Choices

In [5]:
import numpy as np
from dataclasses import dataclass, field
from typing import List, Protocol


# --- 1. Abstracting the Target ---
# This allows you to swap a "Static Point" for a "Moving Circle" easily
class Target(Protocol):
    def update(self, time: float):
        """Returns pos, vel at given time"""
        ...


class StaticTarget:
    def __init__(self, pos: np.ndarray):
        self.pos = pos
        self.vel = np.zeros(3)

    def update(self, time: float):
        return self.pos, self.vel


class CircularTarget:
    def __init__(self, radius=10.0, speed=0.5, height=0.0):
        self.radius = radius
        self.speed = speed
        self.height = height

    def update(self, time: float):
        pos = np.array([
            self.radius * np.cos(self.speed * time),
            self.radius * np.sin(self.speed * time),
            self.height
        ])
        vel = np.array([
            -self.radius * self.speed * np.sin(self.speed * time),
            self.radius * self.speed * np.cos(self.speed * time),
            0.0
        ])
        return pos, vel


# --- 2. The Scenario Container ---
@dataclass
class Scenario:
    name: str
    start_state: State
    target: Target
    obstacles: List[Obstacle] = field(default_factory=list)
    duration: float = 10.0


# --- 3. The Scenario Definitions ---

def get_scenario_1_head_on():
    """Simple static obstacle blocking the path."""
    return Scenario(
        name="Head On Collision Test",
        start_state=State(pos=np.array([0., 0., 0.]), vel=np.array([0., 0., 0.])),
        target=StaticTarget(pos=np.array([10., 0., 0.])),
        obstacles=[
            # Offset slightly Y=0.1 to break symmetry
            Obstacle(np.array([5.0, 0.0, 0.0]), np.array([0., 0., 0.]), radius=1.0)
        ],
        duration=15.0
    )


def get_scenario_2_chase():
    """Chasing a moving target through a patrol."""
    obs_patrol = Obstacle(np.array([5.0, 3.0, 0.0]), np.array([0.0, -1.0, 0.0]), radius=1.0)

    return Scenario(
        name="Circular Chase",
        start_state=State(pos=np.array([0., 0., 0.]), vel=np.array([0., 0., 0.])),
        target=CircularTarget(radius=8.0, speed=0.5),
        obstacles=[obs_patrol],
        duration=25.0
    )


def get_scenario_3_clutter():
    """A field of random static obstacles."""
    # Generate 5 random obstacles
    obs_list = []
    np.random.seed(42)  # Fixed seed for reproducibility
    for _ in range(20):
        center = np.random.uniform(low=[-5, -5, -5], high=[5, 5, 5])
        obs_list.append(Obstacle(center, np.zeros(3), radius=0.8))

    return Scenario(
        name="Cluttered Field",
        start_state=State(pos=np.array([0., 0., 0.]), vel=np.array([0., 0., 0.])),
        target=StaticTarget(pos=np.array([12., -6., 0.])),
        obstacles=obs_list,
        duration=20.0
    )


# Dictionary for easy loading
SCENARIOS = {
    "head_on": get_scenario_1_head_on,
    "chase": get_scenario_2_chase,
    "clutter": get_scenario_3_clutter
}

## Run Simulation

In [6]:
import numpy as np
import plotly.graph_objects as go


def run_simulation(scenario_name="chase", animate=True):
    # 1. Load Scenario
    if scenario_name not in SCENARIOS:
        print(f"Error: Scenario '{scenario_name}' not found.")
        return
    mission = SCENARIOS[scenario_name]()
    print(f"Loaded: {mission.name} | Animate: {animate}")

    # 2. Setup
    dt = 0.01
    drone_physics = DroneDynamics(dt=dt)
    guidance = NominalGuidance()
    safety_filter = SafetyFilter(u_max=10.0)
    current_state = mission.start_state

    # 3. Data Storage
    path_history = []
    target_history = []
    obs_history = []

    total_steps = int(mission.duration / dt)

    # 4. Simulation Loop
    print(f"Simulating {total_steps} steps...")
    for t_step in range(total_steps):
        time = t_step * dt

        # A. Update Scenario (Target + Obstacles)
        target_pos, target_vel = mission.target.update(time)

        current_obs_snapshot = []
        for obs in mission.obstacles:
            if np.linalg.norm(obs.velocity) > 0:
                obs.center += obs.velocity * dt
                # Simple bounce logic
                if abs(obs.center[1]) > 4.0:
                    obs.velocity[1] *= -1
            current_obs_snapshot.append(obs.center.copy())

        obs_history.append(current_obs_snapshot)

        # B. Control & Physics
        u_nom = guidance.compute_u_nom(current_state.pos, current_state.vel, target_pos, target_vel)
        u_safe = safety_filter.filter(u_nom, current_state.pos, current_state.vel, mission.obstacles)
        current_state = drone_physics.step(current_state, u_safe)

        path_history.append(current_state.pos)
        target_history.append(target_pos)

    # 5. Visualization Selector
    path_history = np.array(path_history)
    target_history = np.array(target_history)

    if animate:
        print("Generating Plotly Animation...")
        animate_plotly(path_history, target_history, obs_history, mission, dt)
    else:
        print("Plotting Static Graph...")
        # You can keep your old static plotter or use a plotly static one
        animate_plotly(path_history, target_history, obs_history, mission, dt, static_only=True)




def animate_plotly(path, target_path, obs_history, mission, dt, static_only=False):
    
    total_frames = len(path)
    # Stride: Skip frames to keep animation smooth (approx 150 frames total)
    stride = max(1, total_frames // 150) 
    
    fig = go.Figure()

    # --- HELPER: Sphere Geometry ---
    def get_sphere_data(center, radius):
        u = np.linspace(0, 2 * np.pi, 15)
        v = np.linspace(0, np.pi, 10)
        x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
        y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
        z = center[2] + radius * np.outer(np.ones(np.size(u)), np.cos(v))
        return x, y, z

    # --- STATIC TRACES (Background) ---
    fig.add_trace(go.Scatter3d(
        x=path[:, 0], y=path[:, 1], z=path[:, 2],
        mode='lines', line=dict(color='blue', width=4),
        name='Drone Path', opacity=0.2
    ))

    fig.add_trace(go.Scatter3d(
        x=target_path[:, 0], y=target_path[:, 1], z=target_path[:, 2],
        mode='lines', line=dict(color='green', width=4, dash='dash'),
        name='Target Path', opacity=0.2
    ))

    # --- DYNAMIC TRACES (Actors) ---
    fig.add_trace(go.Scatter3d(
        x=[path[0, 0]], y=[path[0, 1]], z=[path[0, 2]],
        mode='markers', marker=dict(color='blue', size=6),
        name='Drone'
    ))

    fig.add_trace(go.Scatter3d(
        x=[target_path[0, 0]], y=[target_path[0, 1]], z=[target_path[0, 2]],
        mode='markers', marker=dict(color='green', size=8, symbol='diamond'),
        name='Target'
    ))

    obs_start = obs_history[0]
    for i, obs in enumerate(mission.obstacles):
        x_s, y_s, z_s = get_sphere_data(obs_start[i], obs.radius)
        fig.add_trace(go.Surface(
            x=x_s, y=y_s, z=z_s,
            colorscale=[[0, 'red'], [1, 'red']], showscale=False, opacity=0.5,
            name=f'Obs {i}'
        ))

    # --- ANIMATION FRAMES ---
    frames = []
    num_obstacles = len(mission.obstacles)
    dynamic_indices = [2, 3] + list(range(4, 4 + num_obstacles))

    for k in range(0, total_frames, stride):
        frame_data = []
        
        # 1. Update Positions
        drone_pos = path[k]
        target_pos = target_path[k]
        
        frame_data.append(go.Scatter3d(x=[drone_pos[0]], y=[drone_pos[1]], z=[drone_pos[2]])) # Drone
        frame_data.append(go.Scatter3d(x=[target_pos[0]], y=[target_pos[1]], z=[target_pos[2]])) # Target
        
        current_obs = obs_history[k]
        for i in range(num_obstacles):
            x_new, y_new, z_new = get_sphere_data(current_obs[i], mission.obstacles[i].radius)
            frame_data.append(go.Surface(x=x_new, y=y_new, z=z_new))

        # 2. CALCULATE DYNAMIC CAMERA
        # Find the midpoint between drone and target
        mid_x = (drone_pos[0] + target_pos[0]) / 2
        mid_y = (drone_pos[1] + target_pos[1]) / 2
        mid_z = (drone_pos[2] + target_pos[2]) / 2
        
        # Determine "Zoom Level" (Distance between them + Padding)
        dist = np.linalg.norm(drone_pos - target_pos)
        window = max(10.0, dist * 1.5) # Ensure window is at least 10 meters wide
        half_win = window / 2

        # Create the dynamic layout for THIS specific frame
        frame_layout = dict(
            scene=dict(
                xaxis=dict(range=[mid_x - half_win, mid_x + half_win]),
                yaxis=dict(range=[mid_y - half_win, mid_y + half_win]),
                zaxis=dict(range=[mid_z - half_win, mid_z + half_win])
            )
        )

        frames.append(go.Frame(data=frame_data, layout=frame_layout, traces=dynamic_indices, name=str(k)))

    fig.frames = frames

    # --- INITIAL LAYOUT ---
    # We set the initial view to match the first frame's calculation
    start_mid = (path[0] + target_path[0]) / 2
    fig.update_layout(
        width=1000, height=800,
        title=f"Sim: {mission.name} (Follow Camera)",
        scene=dict(
            xaxis=dict(range=[start_mid[0]-10, start_mid[0]+10], title="X"),
            yaxis=dict(range=[start_mid[1]-10, start_mid[1]+10], title="Y"),
            zaxis=dict(range=[start_mid[2]-10, start_mid[2]+10], title="Z"),
            aspectmode='cube'
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        updatemenus=[dict(
            type="buttons",
            buttons=[dict(label="Play",
                          method="animate",
                          args=[None, dict(frame=dict(duration=20, redraw=True), 
                                           fromcurrent=True)])]
        )]
    )

    if static_only:
        fig.show()
    else:
        fig.show()


if __name__ == "__main__":
    run_simulation("head_on", animate=True)
    #run_simulation("chase", animate=True)
    #run_simulation("clutter", animate=True)


Loaded: Head On Collision Test | Animate: True
Simulating 1500 steps...
Generating Plotly Animation...
